O parâmetro 'binary' é usado para determinar se a correlação deve considerar apenas duas classes (por exemplo, 'fácil' e 'difícil') ou múltiplas classes (como 'fácil', 'médio' e 'difícil').

In [ ]:
binary = False

Número de iterações que cada modelo vai fazer na função

In [ ]:
n_iter_DecisionTreeRegressor=50
n_iter_SVR=50
n_iter_NuSVR=50
n_iter_RandomForestRegressor=50
n_iter_XGBRegressor=50

# Modelos de Regressão

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, f1_score, mean_absolute_error, accuracy_score, precision_recall_fscore_support, r2_score
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix

In [ ]:
ind_vars = pd.read_csv(r'..\code_metrics_professor_util.csv', index_col='question')
ind_vars = ind_vars.fillna(0)

ind_vars

In [ ]:

dep_vars = pd.read_csv(r'Correlations_CSV\question_info.csv', index_col='question')

dep_vars = dep_vars.fillna(0)
dep_vars

In [ ]:
dep_vars.describe()

In [ ]:
ind_vars = ind_vars.astype(np.float64)
dep_vars = dep_vars.astype(np.float64)

In [ ]:
import os
import seaborn as sns
import matplotlib.pyplot as plt
import re

def plot_histogram(hst, base_dir="Figures", regression_dir="Regression"):
    """
    Recebe um gráfico de histograma (Axes) e salva em uma pasta específica, dependendo do valor de 'binary'.
    O nome da imagem é baseado no xlabel do gráfico, com espaços convertidos em underscores e caracteres especiais removidos.
    
    Parameters:
        hst (Axes): O objeto de gráfico criado com sns.histplot.
        binary (bool): Se True, o histograma será salvo na subpasta '2'. Se False, será salvo na subpasta '3'.
        base_dir (str): O diretório base para salvar os gráficos. Padrão é "Figures".
        regression_dir (str): O subdiretório base "Regression".
    """
    # Obter o título do gráfico (rótulo de X) e limpar o texto para o nome do arquivo
    xlabel = hst.get_xlabel()  # Obtém o texto do rótulo do eixo X
    filename = re.sub(r'[^a-zA-Z0-9_]', '', xlabel)  # Remove caracteres especiais
    filename = filename.replace(' ', '_')  # Substitui espaços por underscores
    
    # Definir a subpasta (2 ou 3) dependendo de `binary`
    subfolder = "2" if binary else "3"
    
    # Criar diretório para salvar o histograma dentro da subpasta correta
    hist_dir = os.path.join(base_dir, regression_dir, subfolder, "histogram")
    os.makedirs(hist_dir, exist_ok=True)
    
    # Caminho completo do arquivo para salvar
    hist_file_path = os.path.join(hist_dir, f"{filename}.png")

    # Salvar o gráfico
    plt.tight_layout()
    plt.savefig(hist_file_path, bbox_inches='tight')
    print(f"Gráfico do histograma salvo em: {hist_file_path}")
    plt.show()

## Distribuição de frequência

In [ ]:
hst = sns.histplot(data=dep_vars['taxa de erro'] , bins=20)
sns.set_palette('dark')
hst.set_xlabel('Taxa de Erro (%)')
hst.set_ylabel('Frequência')

plot_histogram(hst, base_dir="Figures", regression_dir="Regression")

In [ ]:
hst = sns.histplot(data=dep_vars['tempo_implementacao'], bins=20)
sns.set_palette('dark')
hst.set_xlabel('Tempo de implementação (segundos)')
hst.set_ylabel('Frequência')


plot_histogram(hst, base_dir="Figures", regression_dir="Regression")

In [ ]:
hst = sns.histplot(data=dep_vars['num_eventos'], bins=20)
sns.set_palette('dark')
hst.set_xlabel('Número de eventos')
hst.set_ylabel('Frequência')


plot_histogram(hst, base_dir="Figures", regression_dir="Regression")

In [ ]:
hst = sns.histplot(data=dep_vars['qtd_alteracoes_codigo'], bins=20)
sns.set_palette('dark')
hst.set_xlabel('Quantidade de alterações no código')
hst.set_ylabel('Frequência')


plot_histogram(hst, base_dir="Figures", regression_dir="Regression")

## Funções auxiliares

In [ ]:
# Classificador de questões
def ternary_classify(rows, bounds):
    if len(bounds) != 2:
        raise Exception('quartiles must have 2 values, {} were given'.format(len(bounds)))
        
    values = ('facil', 'medio', 'dificil')

    return rows.apply(lambda row: 
                      values[0] if row <= bounds[0] else
                      values[1] if row <= bounds[1] else
                      values[2]
                     )
    
def get_bounds_ternary(rows, column_name):
    if column_name == 'taxa de erro':
        # Classificação do INEP
        q1 = 20.0  # Primeiro quartil (25%)
        q3 = 40.0 # Terceiro quartil (75%)
        return (q1, q3)
        #return (60.0, 85.0)


    else:
        return np.quantile(rows, q=[1/3, 2/3], method='midpoint')

In [ ]:
def binary_classify(rows, bounds,  custom_classes=None):
    if len(bounds) != 1:
        raise Exception('bounds must have 1 value, {} were given'.format(len(bounds)))
        
    # Classes padrão ou personalizadas
    values = custom_classes if custom_classes else ('facil', 'dificil')

    return rows.apply(lambda row: 
                      values[0] if row <= bounds[0] else
                      values[1])

    
def get_bounds_binary(rows, column_name):
    if column_name == 'taxa de erro':
        # Classificação do INEP
        return (40.0,)
    if column_name == 'discriminacao':
        # Classificação do INEP
        return (0.09,)
    else:
        return np.quantile(rows, q=[0.5], method='midpoint')

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import itertools

def plot_confusion_matrix(cnf_matrix, title, binary=True, base_dir="Figures", regression_dir="Regression", discriminacao=False):
    """
    Plota a matriz de confusão, exibe o gráfico e o salva em uma pasta específica baseada no valor de `binary` e `discriminacao`.

    Parameters:
        cnf_matrix (array-like): A matriz de confusão.
        title (str): O título do gráfico.
        binary (bool): Define a subpasta e o nome do arquivo. True -> Regression/2, False -> Regression/3.
        base_dir (str): O diretório base para salvar os gráficos. Padrão é "Figures".
        regression_dir (str): O subdiretório base "Regression".
        discriminacao (bool): Define se as classes são 'fraco' e 'muito fraco'. Substitui o valor de `binary` se True.
    """
    # Configurar as classes com base nos parâmetros
    if discriminacao:
        classes = ['muito fraco', 'fraco']
        sub_dir = os.path.join(regression_dir, "discriminacao")
        file_name = "matriz_confusao_discriminacao.png"
    else:
        classes = ['fácil', 'difícil'] if binary else ['fácil', 'médio', 'difícil']
        sub_dir = os.path.join(regression_dir, "2" if binary else "3")
        file_name = f"matriz_confusao_{'2' if binary else '3'}.png"

    # Criar o caminho completo
    full_sub_dir = os.path.join(base_dir, sub_dir)
    os.makedirs(full_sub_dir, exist_ok=True)
    
    file_path = os.path.join(full_sub_dir, file_name)

    # Configurar o estilo e criar o gráfico
    plt.figure()
    plt.style.use('default')
    plt.imshow(cnf_matrix, interpolation='nearest', cmap=plt.get_cmap('Blues'))
    plt.title(title)
    plt.colorbar()

    # Configurar os ticks e rótulos
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    # Adicionar valores na matriz
    fmt = 'd'
    thresh = cnf_matrix.max() / 2.
    for i, j in itertools.product(range(cnf_matrix.shape[0]), range(cnf_matrix.shape[1])):
        plt.text(j, i, format(cnf_matrix[i, j], fmt), horizontalalignment="center",
                 color="white" if cnf_matrix[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('Classe predita')
    plt.xlabel('Classe verdadeira')

    # Salvar o gráfico no arquivo especificado
    plt.savefig(file_path, bbox_inches='tight')

    # Informar onde o arquivo foi salvo
    print(f"Gráfico salvo em: {file_path}")

    plt.show()


In [ ]:
def relative_squared_error(y_true, y_pred):
    return np.sum(np.square(np.subtract(y_true,y_pred))) / np.sum(np.square(np.subtract(y_true, np.mean(y_true))))



def relative_absolute_error(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true - np.mean(y_true)))

def r2_adjusted(r2, n, k):
    return 1 - ((1 - r2) * (n - 1)) / (n - k - 1)

In [ ]:
def cross_val_train(ind_vars, y, model, binary_class):    
    # Escolha das funções de classificação
    bounds_fn = get_bounds_binary if binary_class else get_bounds_ternary
    classify_fn = binary_classify if binary_class else ternary_classify
    
    predicted_list = []
    tested_list = []
    
    cat_predicted_list = []
    cat_tested_list = []
    
    n_folds = 4
    kf = KFold(n_folds, shuffle=True, random_state=42)
    for train_index, test_index in kf.split(ind_vars):
        X_train, X_test = ind_vars.iloc[train_index], ind_vars.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]
        
        # Limites e classificação personalizados para 'discriminacao'
        if y.name == 'discriminacao':
            bounds = get_bounds_binary(y_train, y_train.name)
            classify = lambda rows, b: binary_classify(rows, b, custom_classes=('muito fraco', 'fraco'))
        else:
            bounds = bounds_fn(y_train, y_train.name)
            classify = classify_fn
        
        y_true = classify(y_test, bounds)
        
        model.fit(X_train, y_train)
    
        y_pred = pd.Series(model.predict(X_test))
        classified_pred = classify(y_pred, bounds)
        
        predicted_list = np.append(predicted_list, y_pred)
        tested_list = np.append(tested_list, y_test)
        
        cat_predicted_list = np.append(cat_predicted_list, classified_pred)
        cat_tested_list = np.append(cat_tested_list, y_true)
    
    final_scores = {}
    final_scores['mae'] = mean_absolute_error(tested_list, predicted_list)
    final_scores['rae'] = relative_absolute_error(tested_list, predicted_list)
    final_scores['rse'] = relative_squared_error(tested_list, predicted_list)
    final_scores['r2'] = r2_score(tested_list, predicted_list)
    
    # Classes personalizadas apenas para 'discriminacao'
    if y.name == 'discriminacao':
        classes = ('muito fraco', 'fraco')
    else:
        classes = ('facil', 'dificil') if binary_class else ('facil', 'medio', 'dificil')
    
    cnf_matrix = confusion_matrix(cat_tested_list, cat_predicted_list, labels=classes)
        
    return final_scores, cnf_matrix


In [ ]:
def model_train_search_cv(
        df, dep_vars, model, model_class, distributions, n_iter,
        transform_fn=None, transform_by_column=False, binary_class=True
    ):
    # Definindo as colunas para os resultados
    results_col = ['mae', 'rae', 'rse', 'r2']
    
    # Garantindo que o DataFrame 'results' use float64
    results = pd.DataFrame(0, index=dep_vars.columns, columns=results_col, dtype=np.float64)
    cnf_matrixes = {}
    
    # Aplicando a transformação se necessário
    transformed_df = df.copy()
    if transform_fn and not transform_by_column:
        transformed_df = pd.DataFrame(transform_fn(transformed_df))
        
    # Validando o DataFrame transformado
    assert np.isfinite(transformed_df.values).all(), "Dados de entrada contêm valores inválidos (NaN ou Inf)."
    
    # Iterando pelas colunas do DataFrame de variáveis dependentes
    for col in dep_vars.columns:
        if transform_fn and transform_by_column:
            transformed_df = pd.DataFrame(transform_fn(df, col))
        
        # Verificando valores na variável dependente
        assert np.isfinite(dep_vars[col].values).all(), f"A variável dependente '{col}' contém valores inválidos (NaN ou Inf)."
        
        # RandomizedSearchCV para busca de hiperparâmetros
        clf = RandomizedSearchCV(
            estimator=model, 
            param_distributions=distributions, 
            random_state=42, 
            cv=4, 
            n_iter=n_iter, 
            n_jobs=-1,
            scoring='neg_mean_absolute_error',
            return_train_score=True
        )
        
        # Ajustando o modelo
        search = clf.fit(transformed_df, dep_vars[col])
        best_params = {key.split('__')[-1]: value for key, value in search.best_params_.items()}
        print('best params for {}: {}'.format(col, best_params))
        
        # Criando o modelo com os melhores parâmetros
        best_model = model_class(**best_params)
        
        # Treinando com validação cruzada
        metric_train_result, cnf_matrix = cross_val_train(df, dep_vars[col], best_model, binary_class)
        
        # Convertendo explicitamente para float para evitar problemas de compatibilidade
        # Substituindo NaN ou Inf por 0
        results.loc[col] = [float(np.nan_to_num(metric_train_result.get(k, 0), nan=0, posinf=0, neginf=0)) for k in results_col]
        
        cnf_matrixes[col] = cnf_matrix
        
    # Retornando os resultados ordenados e as matrizes de confusão
    return results.sort_values(by=['r2', 'rse'], ascending=False), cnf_matrixes


In [ ]:
def push_results(results, df_results: pd.DataFrame, model):
    results[model] = dict()
    for dep_var, metrics in df_results.iterrows():
        results[model][dep_var] = metrics

In [ ]:
results = dict()

### Árvore de Regressão

In [ ]:
from sklearn.tree import DecisionTreeRegressor

In [ ]:
from sklearn.tree import DecisionTreeRegressor

criterion = ['squared_error', 'friedman_mse', 'absolute_error']
splitter = ['best', 'random']
max_features = ['sqrt', 'log2', None]  # Adicionado 'None'
random_state = [42, 0, 7]  # Adicionados mais valores

# Espaço de busca expandido
distributions = dict(
    criterion=criterion,
    splitter=splitter,
    max_features=max_features,
    random_state=random_state
)

model = DecisionTreeRegressor(random_state=42)
ans, cnf_matrixes_dtr = model_train_search_cv(ind_vars, dep_vars, model, DecisionTreeRegressor, distributions, n_iter_DecisionTreeRegressor, binary_class=binary)
push_results(results, ans, 'Regression Tree')
ans

In [ ]:
cnf_matrixes_dtr['discriminacao']

In [ ]:
cnf_matrixes_dtr['taxa de erro']

In [ ]:
plot_confusion_matrix(cnf_matrixes_dtr['discriminacao'], 'Matriz de Confusão - Discriminação', discriminacao=True)


### SVR

In [ ]:
from sklearn.svm import SVR

In [ ]:
kernel = ['rbf', 'poly']
degree = [2,3]
tol = np.linspace(start=1e-5, stop=1e-3, num=10)
epsilon = np.linspace(start=1e-3, stop=1, num=10)
C = np.linspace(start=1, stop=1000, num=10)

distributions = dict(
    kernel = kernel,
    degree = degree,
    tol = tol,
    epsilon = epsilon,
    C = C,
)
distributions

In [ ]:
model = SVR()

In [ ]:
ans, cnf_matrixes_svr = model_train_search_cv(ind_vars, dep_vars, model, SVR, distributions, n_iter_SVR, binary_class=binary)
push_results(results, ans, 'SVR')
ans

### NuSVR

In [ ]:
from sklearn.svm import NuSVR

In [ ]:
nu = np.linspace(start=0.1, stop=0.5, num=10)
C = np.linspace(start=1, stop=100, num=10)
kernel = ['rbf', 'poly']
degree = [2,3]
         
distributions = dict(
    nu = nu,
    C = C,
    kernel = kernel,
    degree = degree,
)
distributions 

In [ ]:
model = NuSVR()

In [ ]:
ans, cnf_matrixes_nsvr = model_train_search_cv(ind_vars, dep_vars, model, NuSVR, distributions, n_iter_NuSVR, binary_class=binary)
push_results(results, ans, 'NuSVR')
ans

### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

n_estimators = np.linspace(start=100, stop=1000, num=10, dtype=int)
criterion = ['squared_error', 'absolute_error']
max_features = ['sqrt', 'log2', None]
random_state = [42]

distributions = dict(
    n_estimators = n_estimators,
    criterion = criterion,
    max_features = max_features,
    random_state = random_state
)
distributions

In [ ]:
model = RandomForestRegressor()

In [ ]:
ans, cnf_matrixes_rfr = model_train_search_cv(ind_vars, dep_vars, model, RandomForestRegressor, distributions, n_iter_RandomForestRegressor, binary_class=binary)
push_results(results, ans, 'Random Forest')
ans

In [ ]:
cnf_matrixes_rfr

### XGB

In [ ]:
from xgboost import XGBRegressor

In [ ]:
model = XGBRegressor()

In [ ]:
n_estimators = np.linspace(start=100, stop=500, num=10, dtype=int)
max_depth = np.linspace(start=3, stop=15, num=15, dtype=int)
eta = [0.0001, 0.001, 0.01, 0.1, 1.0]
subsample = np.linspace(start=0.1, stop=1.0, num=10, dtype=np.float64)
distributions = dict(n_estimators=n_estimators, max_depth=max_depth, eta=eta, subsample=subsample)
distributions

In [ ]:
ans, cnf_matrixes_xbgr = model_train_search_cv(ind_vars, dep_vars, model, XGBRegressor, distributions, n_iter_XGBRegressor, binary_class=binary)
push_results(results, ans, model='Extreme Gradient Boosting')
ans

## Resultados

In [ ]:
n, k = ind_vars.shape[0], dep_vars.shape[1]
columns = ['classificador', 'r2_adjusted', 'mae', 'rae', 'rse', 'r2']
best_metrics = pd.DataFrame(-np.inf, index=dep_vars.columns, columns=columns)
for model, metrics in results.items():
    for metric, data in metrics.items():
        cur = best_metrics.loc[metric]
        r2_adjusted_score = r2_adjusted(data['r2'], n, k)
        if r2_adjusted_score > cur['r2_adjusted']:
            best_metrics.loc[metric] = model, r2_adjusted_score, *data
best_metrics.sort_values(by=['r2_adjusted'], ascending=False)

In [ ]:
import os

def save_best_metrics_to_csv(best_metrics, base_dir="Regression_CSV", file_name="best_metrics.csv"):
    """
    Salva o DataFrame 'best_metrics' em um arquivo CSV com uma nova coluna 'metricas',
    que conterá os valores do índice atual do DataFrame.

    Parameters:
        best_metrics (pd.DataFrame): O DataFrame contendo as métricas com índice como 'qtd_alteracoes_codigo', etc.
        base_dir (str): O diretório base onde o arquivo CSV será salvo. Padrão é "Regression_CSV".
        file_name (str): O nome do arquivo CSV. Padrão é "best_metrics.csv".
        binary (bool): Determina a subpasta a ser usada. Se True, cria a subpasta "2", se False, cria "3".
    """
    # Determinar a subpasta com base no valor de `binary`
    sub_dir = "2" if binary else "3"

    # Criar o caminho completo
    full_sub_dir = os.path.join(base_dir, sub_dir)
    os.makedirs(full_sub_dir, exist_ok=True)

    # Caminho completo para o arquivo CSV
    csv_file_path = os.path.join(full_sub_dir, file_name)

    # Resetar o índice e transformá-lo na coluna 'metricas'
    best_metrics = best_metrics.reset_index()
    best_metrics.rename(columns={'index': 'metricas'}, inplace=True)

    best_metrics = best_metrics.round(2)

    # Salvar o DataFrame como CSV
    best_metrics.to_csv(csv_file_path, index=False)

    # Imprimir o caminho onde o arquivo foi salvo
    print(f"Arquivo CSV salvo em: {csv_file_path}")


In [ ]:


save_best_metrics_to_csv(best_metrics, base_dir="Regression_CSV", file_name="Best_Metrics_Regression.csv")


In [ ]:
import pandas as pd

def read_best_metrics(file_path):
    """
    Lê o arquivo CSV de métricas e retorna um DataFrame.

    Parameters:
        file_path (str): O caminho completo para o arquivo CSV.

    Returns:
        pd.DataFrame: O DataFrame contendo os dados do arquivo CSV.
    """
    try:
        # Ler o arquivo CSV
        best_metrics = pd.read_csv(file_path)

        return best_metrics
    except FileNotFoundError:
        print(f"Erro: O arquivo {file_path} não foi encontrado.")
    except Exception as e:
        print(f"Erro ao ler o arquivo: {e}")




In [ ]:
file_path = r"Regression_CSV\2\Best_Metrics_Regression.csv"
if os.path.exists(file_path):

    best_metrics_df = read_best_metrics(file_path)
    display(best_metrics_df)

In [ ]:
file_path_2 = r"Regression_CSV\3\Best_Metrics_Regression.csv"
if os.path.exists(file_path_2):
    best_metrics_df = read_best_metrics(file_path_2)
    display(best_metrics_df)


   